# Stage 4 — Phase 2 Fine-tuning (All Folds)

## RUN INSTRUCTIONS
- **Must run AFTER** `phase1_train_all_folds.ipynb` has completed
- Requires `fold_{k}_p1_best.h5` files in `Stage_4/models/`
- **Restart kernel first** to fully reset VRAM before running this notebook

**What this does:** Loads each Phase 1 checkpoint, unfreezes the last 2 backbone layers  
(`conv2d_14` + activation only — avoids conv2d_13 ~400 MB gradient), fine-tunes at low LR.  
Saves `fold_{k}_p2_best.h5` and `fold_{k}_best.h5` (alias used by eval1.ipynb).  

> If OOM persists: change `BATCH_PHASE2 = 2` in the config cell below.

In [1]:
import os
import shutil
import numpy as np
import gc
import json
import warnings
warnings.filterwarnings('ignore')

# ── GPU memory growth — MUST come before any TF/DeepFace call ────────────────
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'Memory growth enabled on {len(gpus)} GPU(s)')

# ── Mixed precision — set ONCE here, never inside the fold loop ───────────────
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print(f'Compute policy: {tf.keras.mixed_precision.global_policy().name}')

from tensorflow.keras.layers import (
    Input, Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
)
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import accuracy_score, precision_score, recall_score
from deepface import DeepFace
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print(f'TensorFlow: {tf.__version__}')
print(f'GPU devices: {len(gpus)}')

Memory growth enabled on 1 GPU(s)
INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU, compute capability 8.6
Compute policy: mixed_float16
TensorFlow: 2.10.0
GPU devices: 1


## Configuration

In [2]:
DATA_DIR     = r'C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\data\imgData\prepImg\step4'
STAGE4_DIR   = r'C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\infant-growth-monitoring-system\mlModels\autisumDetect\sector1\Stage_4'
MODELS_DIR   = os.path.join(STAGE4_DIR, 'models')
FOLDS_DIR    = os.path.join(STAGE4_DIR, 'Folds')
PLOTS_DIR    = os.path.join(STAGE4_DIR, 'plots')
DATA_OUT_DIR = os.path.join(STAGE4_DIR, 'data')

for d in [MODELS_DIR, FOLDS_DIR, PLOTS_DIR, DATA_OUT_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f'Ready: {d}')

SEED            = 123
IMG_SIZE        = (224, 224)
NUM_FOLDS       = 5
# UNFREEZE_LAYERS=2: unfreezes conv2d_14 + activation only.
# conv2d_13 (7x7x512->4096, ~400 MB gradient) stays frozen — safe on 6 GB VRAM.
UNFREEZE_LAYERS = 2
BATCH_PHASE2    = 4    # change to 2 if OOM persists
LR_PHASE2       = 1e-6
EPOCHS_P2       = 20
CLASS_WEIGHT    = {0: 1.0, 1: 2.0}
VGG_MEAN        = [93.5940, 104.7624, 129.1863]

print(f'\nUNFREEZE_LAYERS: {UNFREEZE_LAYERS} | Batch: {BATCH_PHASE2} | '
      f'LR: {LR_PHASE2} | Epochs: {EPOCHS_P2}')

Ready: C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\infant-growth-monitoring-system\mlModels\autisumDetect\sector1\Stage_4\models
Ready: C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\infant-growth-monitoring-system\mlModels\autisumDetect\sector1\Stage_4\Folds
Ready: C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\infant-growth-monitoring-system\mlModels\autisumDetect\sector1\Stage_4\plots
Ready: C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\infant-growth-monitoring-system\mlModels\autisumDetect\sector1\Stage_4\data

UNFREEZE_LAYERS: 2 | Batch: 4 | LR: 1e-06 | Epochs: 20


## Helper Functions

In [3]:
def build_stage4_model(backbone):
    """Identical architecture to ML1.ipynb — DO NOT modify."""
    inputs = Input(shape=(224, 224, 3), name='face_input')
    x = backbone(inputs, training=False)
    x = GlobalAveragePooling2D(name='gap')(x)
    x = Dense(512, activation='relu',
               kernel_regularizer=l2(0.001),
               name='asd_feature_vector_512')(x)
    x = BatchNormalization(name='bn_512')(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu',
               kernel_regularizer=l2(0.001),
               name='asd_feature_vector_256')(x)
    x = Dropout(0.4)(x)
    output = Dense(1, activation='sigmoid', name='classification_output',
                   dtype='float32')(x)
    return Model(inputs=inputs, outputs=output, name='ASD_Stage4')


def compute_f2(precision, recall):
    denom = 4.0 * precision + recall
    if denom == 0:
        return 0.0
    return (5.0 * precision * recall) / denom


def evaluate_on_dataset(model, dataset):
    y_true_list, y_prob_list = [], []
    for imgs, labels in dataset:
        probs = model.predict(imgs, verbose=0).flatten()
        y_prob_list.extend(probs)
        y_true_list.extend(labels.numpy().flatten())
    return np.array(y_true_list), np.array(y_prob_list)


def print_fold_metrics(fold_id, y_true, y_prob, threshold=0.5):
    y_pred = (y_prob > threshold).astype(int)
    y_true = y_true.astype(int)
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    rec  = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    f2   = compute_f2(prec, rec)
    print(f'  Fold {fold_id}: Acc={acc:.4f}  ASD-Recall={rec:.4f}  '
          f'Precision={prec:.4f}  F2={f2:.4f}')
    return acc, prec, rec, f2


print('Helper functions defined.')

Helper functions defined.


## Phase 2 — 5-Fold Fine-tuning Loop

For each fold:
1. Load `fold_{k}_p1_best.h5` weights
2. Unfreeze last `UNFREEZE_LAYERS=2` backbone layers
3. Fine-tune at `LR=1e-6`
4. Save `fold_{k}_p2_best.h5` + `fold_{k}_best.h5` (alias)
5. Evaluate on val set

In [4]:
fold_results = []

for fold_id in range(1, NUM_FOLDS + 1):
    print('\n' + '='*70)
    print(f'  FOLD {fold_id} / {NUM_FOLDS}  —  Phase 2 (fine-tune last {UNFREEZE_LAYERS} layers)')
    print('='*70)

    # ── Verify Phase 1 checkpoint exists ─────────────────────────────────
    p1_path = os.path.join(MODELS_DIR, f'fold_{fold_id}_p1_best.h5')
    if not os.path.exists(p1_path):
        raise FileNotFoundError(
            f'Phase 1 checkpoint not found: {p1_path}\n'
            f'Run phase1_train_all_folds.ipynb first.'
        )

    # ── Load fold splits ──────────────────────────────────────────────────
    train_paths_fold  = np.load(os.path.join(FOLDS_DIR, f'fold_{fold_id}_train_paths.npy'),  allow_pickle=True)
    train_labels_fold = np.load(os.path.join(FOLDS_DIR, f'fold_{fold_id}_train_labels.npy'), allow_pickle=True)
    val_paths_fold    = np.load(os.path.join(FOLDS_DIR, f'fold_{fold_id}_val_paths.npy'),    allow_pickle=True)
    val_labels_fold   = np.load(os.path.join(FOLDS_DIR, f'fold_{fold_id}_val_labels.npy'),   allow_pickle=True)
    print(f'  Train: {len(train_paths_fold)}  Val: {len(val_paths_fold)}')

    # ── Clear session (do NOT call set_global_policy here) ────────────────
    tf.keras.backend.clear_session()
    gc.collect()

    # ── Rebuild augmentation fresh after clear_session ────────────────────
    data_augmentation_fold = tf.keras.Sequential([
        tf.keras.layers.RandomFlip('horizontal'),
        tf.keras.layers.RandomRotation(0.15),
        tf.keras.layers.RandomContrast(0.1),
        tf.keras.layers.RandomZoom(0.1),
    ], name='augmentation')

    def process_image_fold(file_path, label, augment=False):
        img = tf.io.read_file(file_path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, IMG_SIZE)
        img = img[..., ::-1]                        # RGB → BGR
        img = tf.cast(img, tf.float32) - VGG_MEAN  # subtract channel means
        if augment:
            img = tf.expand_dims(img, 0)
            img = data_augmentation_fold(img, training=True)
            img = tf.squeeze(img, 0)
        target = 1.0 - tf.cast(label, tf.float32)  # Autistic=1
        return img, target

    def make_ds(paths, labels, batch_size, augment=False, shuffle=False):
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        if shuffle:
            ds = ds.shuffle(len(paths), seed=SEED)
        ds = ds.map(lambda x, y: process_image_fold(x, y, augment=augment),
                    num_parallel_calls=tf.data.AUTOTUNE)
        return ds.batch(batch_size).prefetch(1)  # prefetch(1) avoids AUTOTUNE over-buffering

    # ── Build datasets ────────────────────────────────────────────────────
    train_ds_p2 = make_ds(train_paths_fold, train_labels_fold,
                           BATCH_PHASE2, augment=True, shuffle=True)
    val_ds_p2   = make_ds(val_paths_fold, val_labels_fold,
                           BATCH_PHASE2, augment=False, shuffle=False)

    # ── Build backbone + model ────────────────────────────────────────────
    vgg_wrapper  = DeepFace.build_model('VGG-Face')
    full_vgg     = vgg_wrapper.model
    backbone_out = full_vgg.get_layer('conv2d_14').output
    backbone     = Model(inputs=full_vgg.input, outputs=backbone_out,
                         name='vgg_face_backbone')
    backbone.trainable = False
    model = build_stage4_model(backbone)

    # ── Load Phase 1 weights ──────────────────────────────────────────────
    # load_weights into same-architecture model keeps backbone reference
    # valid for the unfreezing step that follows.
    print(f'  Loading Phase 1 weights: {p1_path}')
    model.load_weights(p1_path)

    # ── Unfreeze last UNFREEZE_LAYERS backbone layers ─────────────────────
    backbone.trainable = True
    for layer in backbone.layers[:-UNFREEZE_LAYERS]:
        layer.trainable = False
    trainable_count = sum(1 for l in backbone.layers if l.trainable)
    print(f'  Backbone trainable layers: {trainable_count}  '
          f'(UNFREEZE_LAYERS={UNFREEZE_LAYERS})')

    # ── Checkpoint paths ──────────────────────────────────────────────────
    p2_path   = os.path.join(MODELS_DIR, f'fold_{fold_id}_p2_best.h5')
    best_path = os.path.join(MODELS_DIR, f'fold_{fold_id}_best.h5')  # alias for stage4_final_model_evaluation.ipynb

    callbacks_p2 = [
        EarlyStopping(monitor='val_loss', patience=8,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4,
                          verbose=1, min_lr=1e-8),
        ModelCheckpoint(p2_path, monitor='val_loss',
                        save_best_only=True, mode='min', verbose=1),
    ]

    # ── Phase 2 fine-tuning ───────────────────────────────────────────────
    print(f'\n  Phase 2 — fine-tuning last {UNFREEZE_LAYERS} backbone layers  '
          f'(LR={LR_PHASE2}, batch={BATCH_PHASE2})')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR_PHASE2),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.Recall(name='recall')]
    )
    history_p2 = model.fit(
        train_ds_p2,
        validation_data=val_ds_p2,
        epochs=EPOCHS_P2,
        class_weight=CLASS_WEIGHT,
        callbacks=callbacks_p2,
        verbose=1
    )
    print(f'  Checkpoint saved: {p2_path}')

    # Copy as fold_{k}_best.h5 alias
    shutil.copy2(p2_path, best_path)
    print(f'  Alias saved:      {best_path}')

    # ── Training plot ─────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f'Fold {fold_id} — Phase 2 Fine-tuning History', fontsize=13)
    ep = range(1, len(history_p2.history['loss']) + 1)
    axes[0].plot(ep, history_p2.history['loss'],        'r-',  label='train')
    axes[0].plot(ep, history_p2.history['val_loss'],    'r--', label='val')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)
    axes[1].plot(ep, history_p2.history['accuracy'],     'r-',  label='train')
    axes[1].plot(ep, history_p2.history['val_accuracy'], 'r--', label='val')
    axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(True)
    plt.tight_layout()
    plot_path = os.path.join(PLOTS_DIR, f'fold_{fold_id}_p2_history.png')
    plt.savefig(plot_path, dpi=100, bbox_inches='tight')
    plt.close()
    print(f'  Plot saved: {plot_path}')

    # ── Free training model BEFORE loading checkpoint ─────────────────────
    del model, backbone, vgg_wrapper, full_vgg, backbone_out
    del train_ds_p2, val_ds_p2
    del history_p2
    gc.collect()
    tf.keras.backend.clear_session()

    # ── Evaluate on val set ───────────────────────────────────────────────
    print(f'\n  Loading best Phase 2 checkpoint for evaluation...')
    best_model = load_model(p2_path, compile=False)

    # Rebuild augmentation after clear_session (augment=False for eval, so safe)
    data_augmentation_fold = tf.keras.Sequential([
        tf.keras.layers.RandomFlip('horizontal'),
        tf.keras.layers.RandomRotation(0.15),
        tf.keras.layers.RandomContrast(0.1),
        tf.keras.layers.RandomZoom(0.1),
    ], name='augmentation')

    val_ds_eval = make_ds(val_paths_fold, val_labels_fold,
                           batch_size=16, augment=False, shuffle=False)
    y_true_val, y_prob_val = evaluate_on_dataset(best_model, val_ds_eval)

    print(f'\n  ── Fold {fold_id} Results (threshold=0.5) ──')
    acc, prec, rec, f2 = print_fold_metrics(fold_id, y_true_val, y_prob_val)

    fold_results.append({
        'fold':      fold_id,
        'accuracy':  float(acc),
        'precision': float(prec),
        'recall':    float(rec),
        'f2':        float(f2),
    })

    # ── Full cleanup ───────────────────────────────────────────────────────
    del best_model, val_ds_eval, y_true_val, y_prob_val
    gc.collect()
    tf.keras.backend.clear_session()
    print(f'  Fold {fold_id} cleanup done.')

print('\n' + '='*70)
print('  ALL FOLDS COMPLETE')
print('='*70)


  FOLD 1 / 5  —  Phase 2 (fine-tune last 2 layers)
  Train: 2820  Val: 705
  Loading Phase 1 weights: C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\infant-growth-monitoring-system\mlModels\autisumDetect\sector1\Stage_4\models\fold_1_p1_best.h5
  Backbone trainable layers: 2  (UNFREEZE_LAYERS=2)

  Phase 2 — fine-tuning last 2 backbone layers  (LR=1e-06, batch=4)
Epoch 1/20
705/705 [==============================] - ETA: 0s - loss: 2.0118 - accuracy: 0.6404 - recall: 0.8057
Epoch 1: val_loss improved from inf to 1.74705, saving model to C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\infant-growth-monitoring-system\mlModels\autisumDetect\sector1\Stage_4\models\fold_1_p2_best.h5
705/705 [==============================] - 55s 70ms/step - loss: 2.0118 - accuracy: 0.6404 - recall: 0.8057 - val_loss: 1.7470 - val_accuracy: 0.7050 - val_recall: 0.6250 - lr: 1.0000e-06
Epoch 2/20
705/705 [==============================] - ETA: 0s - loss: 1.9752 - accuracy: 0.6606 - recall: 0

## Cross-Validation Summary

In [5]:
df_results = pd.DataFrame(fold_results).set_index('fold')

print('\n' + '='*60)
print('STAGE 4 — 5-FOLD CROSS-VALIDATION SUMMARY (Phase 2)')
print('='*60)
print(df_results.round(4).to_string())
print('-'*60)
for metric in ['accuracy', 'recall', 'f2']:
    m = df_results[metric].mean()
    s = df_results[metric].std()
    label = 'ASD Recall' if metric == 'recall' else metric.capitalize()
    print(f'  Mean {label:12s}: {m:.4f} ± {s:.4f}')
print('='*60)

best_fold = df_results['f2'].idxmax()
print(f'\n  Best fold by F2: Fold {best_fold}  '
      f'(F2={df_results.loc[best_fold,"f2"]:.4f}, '
      f'ASD Recall={df_results.loc[best_fold,"recall"]:.4f})')
print(f'  Best model: {os.path.join(MODELS_DIR, f"fold_{best_fold}_best.h5")}')
print('\n  Note: Select production model by F2-score, NOT accuracy.')

# ── Save results ───────────────────────────────────────────────────────────
csv_path  = os.path.join(DATA_OUT_DIR, 'cv_results_stage4.csv')
json_path = os.path.join(DATA_OUT_DIR, 'cv_results_stage4.json')

df_results.to_csv(csv_path)
with open(json_path, 'w') as f:
    json.dump(fold_results, f, indent=2)

print(f'\n  Results saved: {csv_path}')
print(f'  Results saved: {json_path}')


STAGE 4 — 5-FOLD CROSS-VALIDATION SUMMARY (Phase 2)
      accuracy  precision  recall      f2
fold                                     
1       0.7135     0.7467  0.6449  0.6630
2       0.7333     0.7485  0.7017  0.7106
3       0.7376     0.7546  0.7009  0.7110
4       0.7348     0.7343  0.7322  0.7326
5       0.7078     0.6864  0.7607  0.7446
------------------------------------------------------------
  Mean Accuracy    : 0.7254 ± 0.0137
  Mean ASD Recall  : 0.7081 ± 0.0431
  Mean F2          : 0.7123 ± 0.0312

  Best fold by F2: Fold 5  (F2=0.7446, ASD Recall=0.7607)
  Best model: C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\infant-growth-monitoring-system\mlModels\autisumDetect\sector1\Stage_4\models\fold_5_best.h5

  Note: Select production model by F2-score, NOT accuracy.

  Results saved: C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\infant-growth-monitoring-system\mlModels\autisumDetect\sector1\Stage_4\data\cv_results_stage4.csv
  Results saved: C:\Users\Y